In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adopted P48 joint actual short-tail control

Run the new `local_joint_p48_joint_control_v1_colab.ipynb` with Run all. The task loads ordinary editable source from the existing B development branch. User execution is required; implementation validation does not execute GPU/model/Drive work.

The sole construction corrects the original nonzero bad anchor `A=(.02*d_saved).float()` at saved fork48 with its complete UniPC history. Fixed ZERO/ANCHOR/CANDIDATE rows share one Transformer preparation and one VAE load. The original conditional-clean FP32 adapter precedes CFG5. Both the actual candidate and its difference from A must have L2 norm at most .02, with original latent support. No silent rescale, amplitude scan, re-solve for a better physical direction, P49 fallback or failed-unit replacement exists.

S is Huber deficit h_.01 of the original state correlation gap; P is mean32 h_.02 of all payload margins; W is h_.02 of the all32 minimum margin. All max/min ties use equal active subgradients. Q is the mean of eight energies normalized by their same-run A−Z values: latent outside-S squared L2; RGB outside-R MSE; global MSE; temporal residual MSE; worst-frame MSE; worst-transition MSE; largest squared pixel residual; largest squared temporal pixel residual. R spans the four original ROIs over all 181 frames. Residuals are not clipped. Zero denominators make the construction UNDEFINED, with no epsilon or omitted objective.

Four independent VAE graphs save their RGB and differences from ordinary A. S/P/W use the original pooled-energy DCT reader. Q uses CPU FP64 formulas on the actual FP32 RGB, full RGB cotangents through that same live decoder graph, and the direct latent-outside term. Both terms reach the tail unmasked. The full-history Native48 affine coefficient includes its corrector and predictor, calculated from actual saved sigmas. Final49 is pre-corrector x49−sigma49*v49. Thus each objective uses `k48*(g_z+Jc^T(-5*sigma49*g_z)+Ju^T(4*sigma49*g_z))`, and only this final control cotangent is masked to S. BF16 pullbacks remain approximate path derivatives.

The normalized four control gradients define one max-min SOCP in their span together with A. Two lexicographic numerical stages define one physical candidate: maximize common descent, then minimize correction norm at the recomputed optimum within a declared numerical resolution. Both numerical certificates, FP32 rounding and actual norms/slopes are saved. No resolved positive common descent, zero/nonfinite gradients or uncertain numerical constraints retain C as missing with an explicit reason. A negative or undefined construction is a complete deliverable, not a scientific PASS.

The finite C report retains 22 correlations, all32 margins, 181 frame and180 transition measurements. It separately evaluates improvements against ZERO (S/P, weakest margin, originally positive bits) and against A. Each of the eight `E_k(C)<=.5*E_k(A)` conditions is an adopted construction target, not a perceptual threshold or scientific PASS. No codec is performed.

Independent P48 maximum budget: 1 Transformer preparation;14 DiT forwards (6 ordinary+8 gradient),8 input VJPs,240 original gradient blocks and at most240 block replays;6 native updates;1 VAE load;4 gradient decode/VJP units and3 ordinary decodes;322 original causal chunks and at most184 chunk replays (506 total). P0's1 preparation/2 forwards/2 VJPs/60 replay cost is recorded separately. No new source, 50-step baseline, encode, codec or retry. Staged offload reuses the very same model objects. The four calibration units are not four independent samples.

The 240 tracked original block entries cover only the eight gradient forwards. The six ordinary forwards contain another180 native blocks without checkpoint block instrumentation; all14 forwards comprise420 original blocks. This distinction does not change the replay ceiling.

P48 needs one uninterrupted complete physical run. Each physical call and each tracked block/chunk entry is persisted before execution. Failed/unknown calls consume their allowance. Resource records sample host RSS every .5s, save every60s, record per-stage CUDA allocated/reserved peaks and each spool's storage/traffic, including partial writes. All outputs, cotangents and failed/missing rows remain in the Drive task. After any physical entry, a new Run all restores saved evidence/report only; it does not complete remaining graphs by loading new model objects. A confirmed zero-call saved-source precheck may be corrected and continued into its first physical attempt, preserving earlier failures. File/dtype/shape/finite/support/scheduler checks enforce actual execution semantics; optional source/environment labels do not gate work.

Historical boundary: old TERMINAL46 already ran complete actual tail gradients (2 VAE VJPs and2 tail VJPs;6/6 MP4), but presence C was19/15 versus same-source SINGLE49 30/30. Old MULTI was19/16 and LOCAL16/19. Therefore neither full-tail backprop nor P0 engineering success establishes finite benefit. The new question is joint S/P/W/Q correction at nonzero A, with support leakage and temporal quality in the gradient, measured on one finite C.


In [ ]:
from pathlib import Path
import datetime, importlib, json, os, signal, subprocess, sys, time, traceback
SOURCE_REF = 'dev/local-joint-state-payload-v1'
SOURCE_URL = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/' + SOURCE_REF + '/notebooks/local_joint_p48_joint_control_v1_portable_source.zip'
FIXED_CONFIG = {'model': {'id': 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'}, 'views': ['ZERO', 'ANCHOR', 'CANDIDATE'], 'objectives': ['S', 'P', 'W', 'Q'], 'guidance': 5, 'cap': 0.02, 'block_count': 30, 'transformer_dtype': 'bfloat16', 'device': 'cuda', 'source_run': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Paired-Transfer/resumable-paired-transfer-v1/run', 'spool_root': '/content'}
OUTPUT = Path('/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-P48-Joint-Control/resumable-p48-joint-control-v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
ATTEMPT = OUTPUT / 'notebook_attempts' / STAMP
ATTEMPT.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT = OUTPUT / 'run'
WORKSPACE = Path('/content/Video-WM-P48-' + STAMP)
PYTHON = sys.executable
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n')
    os.replace(tmp, path)
if (OUTPUT / 'fixed_config.json').exists():
    FIXED_CONFIG = json.loads((OUTPUT / 'fixed_config.json').read_text())
else:
    write_json(OUTPUT / 'fixed_config.json', FIXED_CONFIG)
fixed_slots = dict(status='SETUP_STARTED', objectives={n: dict(status='MISSING') for n in ('S','P','W','Q')}, endpoints={b: dict(status='MISSING', reason='not_run') for b in ('ZERO', 'ANCHOR', 'CANDIDATE')}, scientific_pass=False)
write_json(ATTEMPT / 'fixed_slots.json', fixed_slots)
def fail(stage, exc):
    write_json(ATTEMPT / 'failure.json', dict(stage=stage, reason=repr(exc), retry=False, traceback=traceback.format_exc()))
    fixed_slots.update(status='ENGINEERING_FAILURE', reason=repr(exc))
    write_json(ATTEMPT / 'fixed_slots.json', fixed_slots)
def logged(command, stage, *, cwd=None):
    child = None
    try:
        with (ATTEMPT / 'execution.log').open('a') as log:
            print(stage, flush=True)
            child = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                     text=True, bufsize=1, start_new_session=True)
            for line in child.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            return child.wait()
    except BaseException as exc:
        fail(stage, exc)
        raise
    finally:
        if child is not None:
            if child.poll() is None:
                try: os.killpg(child.pid, signal.SIGTERM)
                except ProcessLookupError: pass
                try: child.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    try: os.killpg(child.pid, signal.SIGKILL)
                    except ProcessLookupError: pass
                    child.wait(timeout=10)
            if child.stdout is not None: child.stdout.close()
print('Fixed P48 task:', OUTPUT)

In [ ]:
import io, urllib.request, urllib.error, zipfile
SOURCE_CACHE = OUTPUT / 'source_package.zip'
try:
    downloaded = False
    try:
        with urllib.request.urlopen(SOURCE_URL, timeout=30) as response:
            package = response.read()
        downloaded = True
        source_location = SOURCE_URL
    except Exception as network_error:
        try:
            package = SOURCE_CACHE.read_bytes()
        except OSError as cache_error:
            raise RuntimeError('Source download unavailable: ' + repr(network_error)
                               + '; source cache unreadable: ' + repr(cache_error)) from cache_error
        source_location = str(SOURCE_CACHE)
        print('Network source unavailable; loading saved source ZIP:', repr(network_error))
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(package)) as archive:
        for entry in ('runtime/wan/local_joint_p48_journal_v1.py',
                      'experiments/wan_state_clock/local_joint_p48_joint_control_v1_run.py'):
            try:
                archive.read(entry).decode('utf-8')
            except (KeyError, OSError, ValueError, UnicodeError) as entry_error:
                raise RuntimeError('Source entry unreadable: ' + entry + ': ' + repr(entry_error)) from entry_error
        for member in archive.infolist():
            target = (WORKSPACE / member.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise ValueError('unsafe source ZIP path: ' + member.filename)
            if member.is_dir(): target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
    if downloaded:
        try:
            temporary = SOURCE_CACHE.with_suffix('.zip.tmp')
            temporary.write_bytes(package)
            os.replace(temporary, SOURCE_CACHE)
        except OSError as cache_error:
            print('Source loaded, but saving its offline cache failed:', repr(cache_error))
    for module_name in list(sys.modules):
        if module_name.split('.', 1)[0] in ('main', 'runtime', 'experiments'):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()
    sys.path.insert(0, str(WORKSPACE))
    print('Source location:', source_location, WORKSPACE)
except BaseException as exc:
    fail('SOURCE_LOAD', exc)
    raise

In [ ]:
try:
    from runtime.wan.local_joint_p48_journal_v1 import needs_model_work
    NEEDS_MODEL = needs_model_work(RUN_OUTPUT)
    probe = 'import torch, numpy' + ('; from diffusers import WanTransformer3DModel, AutoencoderKLWan; import cvxpy, clarabel' if NEEDS_MODEL else '')
    rc = logged([PYTHON, '-u', '-c', probe], 'DEPENDENCY_IMPORT')
    if rc:
        packages = ['torch', 'numpy'] + (['diffusers', 'transformers', 'accelerate', 'safetensors', 'huggingface-hub', 'cvxpy', 'clarabel'] if NEEDS_MODEL else [])
        rc = logged([PYTHON, '-m', 'pip', 'install', *packages], 'DEPENDENCY_INSTALL')
        if rc: raise RuntimeError('dependency installation failed; see execution.log')
        if logged([PYTHON, '-u', '-c', probe], 'DEPENDENCY_IMPORT_AFTER_INSTALL'):
            raise RuntimeError('required imports failed; see execution.log')
    if NEEDS_MODEL and str(FIXED_CONFIG.get('device', 'cuda')).startswith('cuda'):
        if logged([PYTHON, '-c', "import torch; assert torch.cuda.is_available(), 'CUDA unavailable for the adopted fixed P48'"], 'CUDA_AVAILABILITY'):
            raise RuntimeError('CUDA unavailable; no model calls attempted')
    try:
        import torch
        write_json(ATTEMPT / 'environment_note.json', dict(python=sys.version, torch=str(torch.__version__), new_model_work=NEEDS_MODEL))
    except Exception as exc: print('Optional environment note unavailable:', repr(exc))
except BaseException as exc:
    fail('DEPENDENCIES', exc)
    raise

In [ ]:
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token: os.environ['HF_TOKEN'] = token
except Exception as exc: print('Optional token unavailable:', type(exc).__name__)
from runtime.wan.local_joint_p48_journal_v1 import finalize_interrupted
rc = None
try:
    rc = logged([PYTHON, '-u', '-m', 'experiments.wan_state_clock.local_joint_p48_joint_control_v1_run',
                 '--config', str(OUTPUT / 'fixed_config.json'), '--output', str(RUN_OUTPUT)], 'P48_FIXED_JOINT_CONTROL', cwd=WORKSPACE)
finally:
    # Child cleanup precedes report-only takeover; never restart a model unit.
    result = finalize_interrupted(RUN_OUTPUT, 'notebook child finished/interrupted; return code ' + str(rc))
    write_json(ATTEMPT / 'execution_receipt.json', dict(returncode=rc, result_status=None if result is None else result['status'], automatic_retry=False))
if result is None: raise RuntimeError('No recoverable P48 result; retain notebook attempt logs')
fixed_slots.update(status=result['status'], endpoints=result['endpoints'], objectives=result['objectives'], counts=result['counts'])
write_json(ATTEMPT / 'fixed_slots.json', fixed_slots)
print('P48 status:', result['status'], 'return code:', rc)
print('Failure/unknown rows remain in the task; no retry.')

In [ ]:
result = json.loads((RUN_OUTPUT / 'result.json').read_text())
print('Status:', result['status'], '; scientific PASS: False')
print('Expected budget:', result['expected_cost'], '; P0 sunk separately:', result['p0_sunk_cost'])
print('Actual calls:', result['counts'], 'blocks:', result['block_counts'], 'chunks:', result['chunk_counts'])
print('Block counting scope:', result['block_count_scope'])
for name, row in result['endpoints'].items(): print(name, row)
for name, row in result['objectives'].items(): print(name, row)
print('SOCP:', result.get('solution')); print('Finite comparison:', result.get('comparison'))
print('Resources:', result['resources']); print('Failures retained:', result['failures'])
print('Current attempt:', result.get('current_attempt')); print(result['ceiling'])
print('Return whole task directory:', OUTPUT)